# Baseline vs Blockade Analyte Signature Analysis

## Setup Instructions

Before running this notebook, set the `ML4NP_DATA_ROOT` environment variable:

```bash
# Linux/Mac
export ML4NP_DATA_ROOT="/mnt/vnas/ml4nanopore_data"

# Windows (PowerShell)
$env:ML4NP_DATA_ROOT="C:\path\to\ml4nanopore_data"
```

In [ ]:
# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import pyabf
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import tsfel
import warnings
import umap
from scipy.optimize import bisect
import shap
import json

In [ ]:
# ============================================================================
# DATA ROOT CONFIGURATION
# ============================================================================

# Get data root from environment variable with fallback
DATA_ROOT_CSV = Path('/mnt/data_share/rushang')
DATA_ROOT = Path(os.environ.get("ML4NP_DATA_ROOT", "/mnt/vnas/ml4nanopore_data"))
print(f"Data root: {DATA_ROOT}")
print(f"Data root exists: {DATA_ROOT.exists()}")

if not DATA_ROOT.exists():
    print("\nWARNING: Data root directory does not exist!")
    print("Please set ML4NP_DATA_ROOT environment variable to your data directory.")
    print("Example: export ML4NP_DATA_ROOT=/mnt/vnas/ml4nanopore_data")

# Create output directory
output_dir = Path("analyte_signature_complete_analysis")
output_dir.mkdir(exist_ok=True)
print(f"Output directory: {output_dir.absolute()}")

In [ ]:
# ============================================================================
# FILE PATHS CONFIGURATION
# ============================================================================

# Define file pairs with paths relative to DATA_ROOT
file_pairs = [
    {
        'classification_csv': DATA_ROOT_CSV / "classified/20200309_wtAeL_4M_KCl_1L_1X_L2AS4_AS5_AS6_AS7_AS8_AS9_AS10.abf_currentlevels_classified.csv",
        'abf_file': DATA_ROOT / "raw/PeptideLadders/Ladder2/20200309 wtAeL 4M KCl 1µL 1X L2AS4_AS5_AS6_AS7_AS8_AS9_AS10.abf"
    },
    # Add more files as needed:
    # {
    #     'classification_csv': DATA_ROOT / "classified/2020-03-13_wtAeL_L1AS4_AS5_AS6_AS7_AS8_AS9_AS10.abf_currentlevels_classified.csv",
    #     'abf_file': DATA_ROOT / "raw/PeptideLadders/Ladder1/2020-03-13 wtAeL L1AS4 AS5 AS6 AS7 AS8 AS9 AS10.abf"
    # },
]

# Verify files exist
print("\nChecking file availability:")
for i, pair in enumerate(file_pairs):
    print(f"\nPair {i+1}:")
    csv_exists = pair['classification_csv'].exists()
    abf_exists = pair['abf_file'].exists()
    print(f"  CSV: {'✓' if csv_exists else '✗'} - {pair['classification_csv']}")
    print(f"  ABF: {'✓' if abf_exists else '✗'} - {pair['abf_file']}")
    if not (csv_exists and abf_exists):
        print(f"WARNING: Some files missing for pair {i+1}")

In [ ]:
# ============================================================================
# SAMPLING PARAMETERS
# ============================================================================

RANDOM_STATE = 42
MAX_EVENT_LENGTH = 1000
BASELINE_SEGMENT_LENGTH = 1000
MAX_BASELINE_SEGMENTS_PER_FILE = 2000

print("\nSampling Parameters:")
print(f"Random State: {RANDOM_STATE}")
print(f"Max Event Length: {MAX_EVENT_LENGTH}")
print(f"Baseline Segment Length: {BASELINE_SEGMENT_LENGTH}")
print(f"Max Baseline Segments per File: {MAX_BASELINE_SEGMENTS_PER_FILE}")

In [ ]:
# ============================================================================
# CORE FUNCTIONS
# ============================================================================

def soft_threshold(x, delta):
    """Soft-thresholding operator"""
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0)

def sparse_kmeans(X, K, s, true_labels=None, max_iter=1000, tol=1e-6, patience=50, min_iter=10, random_state=42):
    """
    Sparse K-means clustering following Witten & Tibshirani (2010)
    """
    n, p = X.shape
    
    if s < 1 or s > np.sqrt(p):
        raise ValueError(f"s must be between 1 and sqrt(p)={np.sqrt(p):.2f}")
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    weights = np.ones(p) / np.sqrt(p)
    objective_history = []
    weight_history = [weights.copy()]
    best_weights = weights.copy()
    best_objective = -np.inf
    patience_counter = 0
    converged = False
    
    print(f"{n} samples, {p} features, s={s}")
    
    for iteration in range(max_iter):
        weighted_X = X_scaled * np.sqrt(weights)
        kmeans = KMeans(n_clusters=K, random_state=random_state, n_init=10)
        cluster_labels = kmeans.fit_predict(weighted_X)
        
        BCSS = np.zeros(p)
        for j in range(p):
            overall_mean = np.mean(X_scaled[:, j])
            total_ss = np.sum((X_scaled[:, j] - overall_mean) ** 2)
            within_ss = 0
            for k in range(K):
                cluster_indices = (cluster_labels == k)
                n_k = np.sum(cluster_indices)
                if n_k > 0:
                    cluster_mean = np.mean(X_scaled[cluster_indices, j])
                    within_ss += np.sum((X_scaled[cluster_indices, j] - cluster_mean) ** 2)
            BCSS[j] = (total_ss - within_ss)
        
        a = BCSS / (np.linalg.norm(BCSS, 2) + 1e-10)
        
        def l1_norm(delta):
            w_temp = soft_threshold(a, delta)
            norm_w = np.linalg.norm(w_temp, 2)
            if norm_w > 0:
                w_temp = w_temp / norm_w
            return np.sum(np.abs(w_temp)) - s
        
        current_l1 = np.sum(np.abs(a / (np.linalg.norm(a, 2) + 1e-10)))
        
        if current_l1 <= s:
            delta = 0
            w_new = a
        else:
            delta_low, delta_high = 0, np.max(np.abs(a))
            delta = bisect(l1_norm, delta_low, delta_high, xtol=1e-8, maxiter=50)
            w_new = soft_threshold(a, delta)
        
        norm_w = np.linalg.norm(w_new, 2)
        if norm_w > 0:
            w_new = w_new / norm_w
        else:
            w_new = np.ones(p) / np.sqrt(p)
        
        current_objective = np.sum(w_new * BCSS)
        objective_history.append(current_objective)
        weight_history.append(w_new.copy())

        if current_objective > best_objective:
            best_objective = current_objective
            best_weights = w_new.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        weight_change = np.linalg.norm(w_new - weights, 2)
        n_nonzero = np.sum(w_new > 1e-6)
        
        if (iteration + 1) % 10 == 0:
            print(f"Iteration {iteration + 1}:")
            print(f"  - Objective: {current_objective:.6f}")
            print(f"  - Non-zero features: {n_nonzero}/{p} ({n_nonzero/p*100:.1f}%)")
            print(f"  - Weight change: {weight_change:.6f}")
        
        if weight_change < tol and iteration >= min_iter:
            print(f"✓ Converged after {iteration + 1} iterations (weight change < {tol})")
            converged = True
            break
        elif patience_counter >= patience and iteration >= min_iter:
            print(f"⚠️ Early stopping after {iteration + 1} iterations (no improvement for {patience} iterations)")
            converged = True
            break
            
        weights = w_new
    
    if not converged:
        print(f"✗ Reached maximum iterations ({max_iter})")
    
    final_weights = best_weights
    final_n_nonzero = np.sum(final_weights > 1e-6)
    
    print(f"\nFinal: {final_n_nonzero} non-zero features, Best objective: {best_objective:.6f}")
    
    return cluster_labels, final_weights, objective_history

def extract_baseline_segments(abf_data, event_regions, num_segments=100):
    """Extract baseline segments avoiding event regions"""
    baseline_segments = []
    total_length = len(abf_data.data[0])
    
    for attempt in range(num_segments * 10):
        start_idx = np.random.randint(0, total_length - BASELINE_SEGMENT_LENGTH)
        end_idx = start_idx + BASELINE_SEGMENT_LENGTH
        
        overlap = False
        for event_start, event_end in event_regions:
            if not (end_idx < event_start or start_idx > event_end):
                overlap = True
                break
        
        if not overlap:
            baseline_signal = abf_data.data[0][start_idx:end_idx]
            baseline_segments.append(baseline_signal)
            
        if len(baseline_segments) >= num_segments:
            break
            
    return baseline_segments

def extract_tsfel_features(signal, segment_id):
    """Extract features with tsfel"""
    try:
        if len(signal) < 20:
            return None
 
        df = pd.DataFrame({"signal": signal})
        cfg = tsfel.get_features_by_domain()
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            features = tsfel.time_series_features_extractor(cfg, df, verbose=0)
        
        if features is None or features.empty:
            return None
        
        features = features.reset_index(drop=True)
        features['segment_id'] = segment_id
        
        return features
        
    except Exception as e:
        return None

def create_balanced_dataset_for_abf(abf_path, classification_csv_path):
    """
    Create balanced dataset with baseline and event segments for one ABF file
    """
    try:
        classified_data = pd.read_csv(classification_csv_path)
    except Exception as e:
        print(f"Error reading CSV {classification_csv_path}: {e}")
        return None
    
    print(f"\nProcessing: {os.path.basename(str(abf_path))}")
    print(f"  Loaded {len(classified_data)} events from CSV")
    
    if len(classified_data) == 0:
        print(f"No matching events found")
        return None
        
    try:
        abf_data = pyabf.ABF(str(abf_path))
        print(f"Successfully loaded ABF file")
    except Exception as e:
        print(f"Error loading ABF file: {e}")
        return None
    
    event_regions = []
    for _, row in classified_data.iterrows():
        event_regions.append((int(row["idxstart"]), int(row["idxend"])))
    
    num_baseline_segments = min(len(classified_data), MAX_BASELINE_SEGMENTS_PER_FILE)
    baseline_segments = extract_baseline_segments(abf_data, event_regions, num_baseline_segments)
    
    print(f"  Extracted {len(baseline_segments)} baseline segments")
    
    all_features = []
    baseline_count = 0
    
    # Extract baseline features
    for i, baseline_signal in enumerate(baseline_segments):
        features = extract_tsfel_features(baseline_signal, f"baseline_{i}")
        if features is not None and not features.empty:
            features['true_label'] = f"baseline"
            features['analyte_label'] = f"baseline"
            features['abf_file'] = os.path.basename(str(abf_path))
            features['segment_type'] = 'baseline'
            features['segment_length'] = len(baseline_signal)
            all_features.append(features)
            baseline_count += 1
    
    # Extract event features
    event_count = 0
    for i, row in classified_data.iterrows():
        s, e = int(row["idxstart"]), int(row["idxend"])
        event_signal = abf_data.data[0][s:e]
        
        features = extract_tsfel_features(event_signal, f"event_{i}")
        if features is not None and not features.empty:
            features['true_label'] = row['final_label']
            features['analyte_label'] = row['final_label']
            features['abf_file'] = os.path.basename(str(abf_path))
            features['segment_type'] = 'event'
            features['segment_length'] = len(event_signal)
            features['log_length'] = np.log(len(event_signal))
            all_features.append(features)
            event_count += 1
    
    print(f"  Processed: {baseline_count} baseline + {event_count} event segments")
    
    if all_features:
        combined_features = pd.concat(all_features, ignore_index=True)
        print(f"  Total features shape: {combined_features.shape}")
        return combined_features
    else:
        print(f"No features extracted")
        return None

def stratified_sample_by_analyte(feature_df, max_per_analyte=2000):
    """
    Stratified sampling to balance analytes and prevent over-representation
    """
    sampled_dfs = []
    
    for analyte in feature_df['analyte_label'].unique():
        analyte_data = feature_df[feature_df['analyte_label'] == analyte]
        n_sample = min(len(analyte_data), max_per_analyte)
        sampled = analyte_data.sample(n=n_sample, random_state=RANDOM_STATE)
        sampled_dfs.append(sampled)
    
    return pd.concat(sampled_dfs, ignore_index=True)

In [ ]:
# ============================================================================
# ANALYSIS FUNCTIONS
# ============================================================================

def perform_clustering_analysis(feature_df):
    """
    Clustering to discover natural structure
    """
    print("\n" + "="*60)
    print("EXPLORATORY CLUSTERING ANALYSIS")
    print("="*60)
    
    X = feature_df.select_dtypes(include=[np.number])
    X = X.loc[:, X.notna().all(axis=0)]
    X = X.loc[:, X.var(axis=0) > 1e-12]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    true_labels = feature_df['analyte_label']
    le = LabelEncoder()
    y_true = le.fit_transform(true_labels)
    n_true_clusters = len(le.classes_)
    
    print(f"True classes: {list(le.classes_)}")
    print(f"Number of samples: {len(X_scaled)}")
    print(f"Number of features: {X_scaled.shape[1]}")
    print(f"Performing sparse K-means with {n_true_clusters} clusters...")
    
    cluster_labels, weights, objective_history = sparse_kmeans(
        X=X_scaled,
        K=n_true_clusters,
        s=2.0,
        max_iter=100,
        random_state=RANDOM_STATE
    )
    
    # Calculate clustering accuracy
    cluster_to_true_label = {}
    for cluster in np.unique(cluster_labels):
        cluster_indices = np.where(cluster_labels == cluster)[0]
        true_labels_in_cluster = y_true[cluster_indices]
        most_common_true_label = np.bincount(true_labels_in_cluster).argmax()
        cluster_to_true_label[cluster] = most_common_true_label

    correct_predictions = 0
    total_predictions = len(y_true)
    for cluster, true_label in cluster_to_true_label.items():
        cluster_indices = np.where(cluster_labels == cluster)[0]
        predicted_labels_in_cluster = cluster_labels[cluster_indices]
        correct_predictions += np.sum(predicted_labels_in_cluster == true_label)

    accuracy = correct_predictions / total_predictions
    print(f"\nClustering accuracy: {accuracy:.3f}")

    for cluster, true_label in cluster_to_true_label.items():
        cluster_indices = np.where(cluster_labels == cluster)[0]
        true_labels_in_cluster = y_true[cluster_indices]
        cluster_accuracy = np.sum(true_labels_in_cluster == true_label) / len(true_labels_in_cluster)
        print(f"  Cluster {cluster} (mapped to {le.classes_[true_label]}): {cluster_accuracy:.3f} accuracy")

    # Feature importance visualization
    feature_names = X.columns
    sorted_indices = np.argsort(weights)[::-1][:10]
    sorted_weights = weights[sorted_indices]
    sorted_feature_names = np.array(feature_names)[sorted_indices]

    plt.figure(figsize=(10, 6))
    plt.barh(range(len(sorted_feature_names)), sorted_weights)
    plt.yticks(range(len(sorted_feature_names)), sorted_feature_names)
    plt.xlabel('Feature Weight')
    plt.title('Top 10 Feature Weights from Sparse K-means')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    # UMAP visualization
    plt.figure(figsize=(12, 5))
    
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
    X_umap = reducer.fit_transform(X_scaled)
    
    plt.subplot(1, 2, 1)
    scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_true, cmap='tab10', alpha=0.6, s=10)
    handles, _ = scatter.legend_elements()
    labels = [le.classes_[i] for i in range(len(le.classes_))]
    plt.legend(handles, labels, title='True Labels', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title('True Labels (UMAP)')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    
    plt.subplot(1, 2, 2)
    scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels, cmap='tab10', alpha=0.6, s=10)
    plt.title('K-means Clusters (UMAP)')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'clustering_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

    ari = adjusted_rand_score(y_true, cluster_labels)
    nmi = normalized_mutual_info_score(y_true, cluster_labels)
    print(f"\nClustering Metrics:")
    print(f"  Adjusted Rand Index (ARI): {ari:.3f}")
    print(f"  Normalized Mutual Information (NMI): {nmi:.3f}")
    
    return {
        'cluster_labels': cluster_labels,
        'true_labels': y_true,
        'feature_matrix': X_scaled,
        'feature_names': X.columns.tolist(),
        'weights': weights,
        'ari': ari,
        'nmi': nmi,
        'accuracy': accuracy
    }

def perform_supervised_analysis(feature_df):
    """
    Supervised analysis to identify discriminative features
    """
    print("\n" + "="*60)
    print("SUPERVISED FEATURE ANALYSIS")
    print("="*60)
    
    X = feature_df.select_dtypes(include=[np.number])
    X = X.loc[:, X.notna().all(axis=0)]
    X = X.loc[:, X.var(axis=0) > 1e-12]
    
    # Task 1: Event vs Baseline (binary)
    y_binary = (feature_df['segment_type'] == 'event').astype(int)
    
    # Task 2: Multi-class analyte discrimination
    le = LabelEncoder()
    y_multiclass = le.fit_transform(feature_df['analyte_label'])
    class_names = le.classes_
    
    print(f"Binary classification: {np.sum(y_binary==0)} baseline vs {np.sum(y_binary==1)} events")
    print(f"Multi-class: {len(class_names)} classes - {list(class_names)}")
    print(f"Features: {X.shape[1]}")
    
    # Binary classification
    print("\n1. Binary Classification (Event vs Baseline):")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=0.3, random_state=RANDOM_STATE, stratify=y_binary
    )
    
    rf_binary = RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    rf_binary.fit(X_train, y_train)
    y_pred_binary = rf_binary.predict(X_test)
    binary_accuracy = accuracy_score(y_test, y_pred_binary)
    
    from sklearn.model_selection import cross_val_score
    cv_scores = cross_val_score(rf_binary, X, y_binary, cv=5)
    
    print(f"  Accuracy: {binary_accuracy:.3f}")
    print(f"  Cross-validation: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    
    binary_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_binary.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n  Top 10 features for Event vs Baseline:")
    for i, row in binary_importance.head(10).iterrows():
        print(f"    {row['feature']}: {row['importance']:.4f}")
    
    # Multi-class classification
    print("\n2. Multi-class Analyte Discrimination:")
    X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
        X, y_multiclass, test_size=0.3, random_state=RANDOM_STATE, stratify=y_multiclass
    )
    
    rf_multi = RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    rf_multi.fit(X_train_multi, y_train_multi)
    y_pred_multi = rf_multi.predict(X_test_multi)
    multi_accuracy = accuracy_score(y_test_multi, y_pred_multi)
    
    print(f"  Accuracy: {multi_accuracy:.3f}")
    
    multi_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_multi.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n  Top 10 features for analyte discrimination:")
    for i, row in multi_importance.head(10).iterrows():
        print(f"    {row['feature']}: {row['importance']:.4f}")
    
    # Visualizations
    plt.figure(figsize=(15, 5))
    
    # Plot 1: Binary feature importance
    plt.subplot(1, 3, 1)
    top_binary = binary_importance.head(10)
    plt.barh(range(len(top_binary)), top_binary['importance'].values)
    plt.yticks(range(len(top_binary)), [f[:20] + '...' if len(f) > 20 else f for f in top_binary['feature']])
    plt.title('Top Features: Event vs Baseline')
    plt.xlabel('Importance')
    
    # Plot 2: Multi-class feature importance
    plt.subplot(1, 3, 2)
    top_multi = multi_importance.head(10)
    plt.barh(range(len(top_multi)), top_multi['importance'].values)
    plt.yticks(range(len(top_multi)), [f[:20] + '...' if len(f) > 20 else f for f in top_multi['feature']])
    plt.title('Top Features: Analyte Discrimination')
    plt.xlabel('Importance')
    
    # Plot 3: Confusion matrix for multi-class
    plt.subplot(1, 3, 3)
    cm = confusion_matrix(y_test_multi, y_pred_multi)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Multi-class Confusion Matrix\n(Accuracy: {multi_accuracy:.3f})')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'supervised_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {
        'binary_accuracy': binary_accuracy,
        'multi_accuracy': multi_accuracy,
        'binary_importance': binary_importance,
        'multi_importance': multi_importance,
        'cv_scores': cv_scores,
        'class_names': class_names.tolist()
    }

def compare_clustering_supervised_results(clustering_results, supervised_results):
    """
    Compare unsupervised clustering vs supervised results
    """
    print("\n" + "="*60)
    print("METHOD COMPARISON")
    print("="*60)
    
    clustering_features = clustering_results['feature_names']
    supervised_binary_top = supervised_results['binary_importance'].head(20)['feature'].values
    supervised_multi_top = supervised_results['multi_importance'].head(20)['feature'].values
    
    binary_overlap = len(set(clustering_features) & set(supervised_binary_top))
    multi_overlap = len(set(clustering_features) & set(supervised_multi_top))
    
    print(f"Features in top 20 binary importance also in clustering: {binary_overlap}/20")
    print(f"Features in top 20 multi-class importance also in clustering: {multi_overlap}/20")
    
    # Create comparison visualization
    plt.figure(figsize=(12, 8))
    
    # Plot 1: Performance metrics
    plt.subplot(2, 2, 1)
    methods = ['Clustering ARI', 'Clustering NMI', 'Binary RF', 'Multi-class RF']
    scores = [clustering_results['ari'], clustering_results['nmi'], 
              supervised_results['binary_accuracy'], supervised_results['multi_accuracy']]
    colors = ['lightblue', 'lightblue', 'lightgreen', 'lightcoral']
    bars = plt.bar(methods, scores, color=colors)
    plt.ylim(0, 1)
    plt.title('Performance Comparison')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{score:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Plot 2: Feature overlap
    plt.subplot(2, 2, 2)
    overlap_types = ['Binary Overlap', 'Multi-class Overlap']
    overlap_counts = [binary_overlap, multi_overlap]
    bars = plt.bar(overlap_types, overlap_counts, color=['orange', 'purple'])
    plt.ylim(0, 20)
    plt.title('Feature Importance Overlap\n(Top 20 features)')
    plt.ylabel('Number of Overlapping Features')
    for bar, count in zip(bars, overlap_counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
                f'{count}/20', ha='center', va='bottom')
    
    # Plot 3: Cross-validation scores
    plt.subplot(2, 2, 3)
    cv_scores = supervised_results['cv_scores']
    plt.boxplot(cv_scores)
    plt.scatter([1]*len(cv_scores), cv_scores, alpha=0.5, color='red')
    plt.title('Binary Classification CV Scores')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)
    
    # Plot 4: Summary
    plt.subplot(2, 2, 4)
    metrics = ['Clustering\nAccuracy', 'Binary\nAccuracy', 'Multi-class\nAccuracy']
    values = [clustering_results['accuracy'], supervised_results['binary_accuracy'], supervised_results['multi_accuracy']]
    bars = plt.bar(metrics, values, color=['skyblue', 'lightgreen', 'salmon'])
    plt.ylim(0, 1)
    plt.title('Accuracy Comparison')
    plt.ylabel('Accuracy')
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{value:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'method_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nSummary Findings:")
    print(f"- Clustering ARI: {clustering_results['ari']:.3f}")
    print(f"- Clustering NMI: {clustering_results['nmi']:.3f}")
    print(f"- Binary classification accuracy: {supervised_results['binary_accuracy']:.3f}")
    print(f"- Multi-class accuracy: {supervised_results['multi_accuracy']:.3f}")

In [ ]:
# ============================================================================
# MAIN PIPELINE EXECUTION
# ============================================================================

print("="*60)
print("BASELINE VS BLOCKADE ANALYTE SIGNATURE ANALYSIS")
print("="*60)
print(f"Data Root: {DATA_ROOT}")
print(f"Random State: {RANDOM_STATE}")
print(f"Output Directory: {output_dir}")
print("="*60)

# Step 1: Create balanced dataset
print("\nSTEP 1: Creating balanced dataset across all ABF files")
all_features = []

for file_pair in tqdm(file_pairs, desc="Processing ABF files"):
    features = create_balanced_dataset_for_abf(
        file_pair['abf_file'],
        file_pair['classification_csv']
    )
    
    if features is not None:
        all_features.append(features)

if not all_features:
    print("\nNo features extracted from any files. Exiting.")
    raise SystemExit("No data available for analysis.")

combined_features = pd.concat(all_features, ignore_index=True)
print(f"\nCombined dataset: {len(combined_features)} total segments")
print(f"\nClass distribution:")
print(combined_features['analyte_label'].value_counts())

In [ ]:
# Step 2: Data cleaning and preprocessing
print("\nSTEP 2: Data cleaning and preprocessing")

# Drop features with >30% NaN values
nan_threshold = 0.3 * len(combined_features)
combined_features.replace([np.inf, -np.inf], np.nan, inplace=True)

final_feature_df = combined_features.dropna(axis=1, thresh=nan_threshold)
print(f"  After dropping features with >30% NaN: {final_feature_df.shape}")

# Remove specific feature types
final_feature_df = final_feature_df.loc[:, ~final_feature_df.columns.str.contains('MFCC')]
final_feature_df = final_feature_df.loc[:, ~final_feature_df.columns.str.contains('signal_ECDF')]
print(f"  After removing MFCC and ECDF features: {final_feature_df.shape}")

# Drop specific columns
cols_to_drop = [
    'signal_Interquartile range', 'signal_Min', 'signal_Max',
    'signal_Median frequency', 'signal_Centroid', 'signal_Spectral centroid',
    'signal_Standard deviation', 'signal_Variance',
    "signal_Root mean square", "signal_Average power",
    "signal_Histogram mode", "signal_Mean", "signal_Median"
]

cols_dropped = []
for col in cols_to_drop:
    if col in final_feature_df.columns:
        final_feature_df = final_feature_df.drop(columns=[col])
        cols_dropped.append(col)
        
print(f"  Dropped {len(cols_dropped)} specific features")

# Process labels
print("\n  Processing labels...")
final_feature_df.loc[~final_feature_df['analyte_label'].str.startswith('b', na=False), 'analyte_label'] = 'event'
final_feature_df.loc[~final_feature_df['true_label'].str.startswith('b', na=False), 'true_label'] = 'event'

print(f"  Final dataset shape: {final_feature_df.shape}")
print(f"  Analyte distribution:")
print(final_feature_df['analyte_label'].value_counts())

In [ ]:
# Step 3: Stratified sampling
print("\nSTEP 3: Stratified sampling for balanced analysis")
balanced_features = stratified_sample_by_analyte(final_feature_df, max_per_analyte=3000)
print(f"  After sampling: {len(balanced_features)} segments")
print(f"  Balanced distribution:")
print(balanced_features['analyte_label'].value_counts())

# Save the balanced dataset
balanced_features.to_csv(output_dir / 'balanced_analyte_features.csv', index=False)
print(f"Saved balanced dataset to: {output_dir / 'balanced_analyte_features.csv'}")

In [ ]:
# Step 4: Perform analyses
print("\nSTEP 4: Performing analyses")

# Clustering analysis
clustering_results = perform_clustering_analysis(balanced_features)

# Supervised analysis
supervised_results = perform_supervised_analysis(balanced_features)

# Compare results
compare_clustering_supervised_results(clustering_results, supervised_results)

In [ ]:
# Step 5: Save results
print("\nSTEP 5: Saving results")

# Save feature importance
supervised_results['binary_importance'].to_csv(output_dir / 'binary_feature_importance.csv', index=False)
supervised_results['multi_importance'].to_csv(output_dir / 'multiclass_feature_importance.csv', index=False)
print(f"Saved feature importance files")

# Save summary
summary = {
    'data_root': str(DATA_ROOT),
    'random_state': RANDOM_STATE,
    'total_segments': len(balanced_features),
    'n_abf_files': len(file_pairs),
    'clustering_ari': float(clustering_results['ari']),
    'clustering_nmi': float(clustering_results['nmi']),
    'clustering_accuracy': float(clustering_results['accuracy']),
    'binary_accuracy': float(supervised_results['binary_accuracy']),
    'multiclass_accuracy': float(supervised_results['multi_accuracy']),
    'cv_mean_accuracy': float(supervised_results['cv_scores'].mean()),
    'cv_std_accuracy': float(supervised_results['cv_scores'].std()),
    'analytes_analyzed': balanced_features['analyte_label'].unique().tolist(),
    'timestamp': pd.Timestamp.now().isoformat(),
    'parameters': {
        'max_event_length': MAX_EVENT_LENGTH,
        'baseline_segment_length': BASELINE_SEGMENT_LENGTH,
        'max_baseline_segments': MAX_BASELINE_SEGMENTS_PER_FILE
    }
}

with open(output_dir / 'analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Saved analysis summary")
print(f"\nAnalysis complete! All results saved to: {output_dir.absolute()}")
print("="*60)

In [ ]:
# ============================================================================
# OPTIONAL: SHAP ANALYSIS
# ============================================================================

print("\nOPTIONAL: SHAP Analysis")

# Get only signal features
signal_cols = [col for col in balanced_features.columns if col.startswith('signal_')]
X = balanced_features[signal_cols].copy()
y = balanced_features['analyte_label']

# Clean data
y = y.astype(str).str.strip()
valid_labels = ['baseline', 'event']
mask = y.isin(valid_labels)
X = X[mask]
y = y[mask]

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
X = X.select_dtypes(include=[np.number])
X = X.fillna(0)

print(f"Data for SHAP: X shape={X.shape}, y unique={y.unique().tolist()}")

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
y_encoded = LabelEncoder().fit_transform(y)
model.fit(X, y_encoded)

# SHAP with sampling
sample_size = min(500, len(X))
sample_idx = np.random.choice(len(X), sample_size, replace=False)
X_sample = X.iloc[sample_idx]

try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # Plot SHAP values
    plt.figure(figsize=(16, 10))
    
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap.summary_plot(shap_values[1], X_sample, show=False, max_display=20)
    elif hasattr(shap_values, 'shape') and len(shap_values.shape) == 3:
        shap.summary_plot(shap_values[:, :, 1], X_sample, show=False, max_display=20)
    else:
        shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
    
    plt.title("SHAP Values - Events vs Baseline", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(left=0.3)
    plt.savefig(output_dir / 'shap_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("SHAP analysis completed and saved")
    
except Exception as e:
    print(f"SHAP failed: {e}")
    print("Using feature importance instead...")
    
    importance = model.feature_importances_
    top_idx = np.argsort(importance)[-15:][::-1]
    
    plt.figure(figsize=(10, 8))
    plt.barh(range(15), importance[top_idx])
    plt.yticks(range(15), [X.columns[i] for i in top_idx])
    plt.gca().invert_yaxis()
    plt.xlabel('Feature Importance')
    plt.title('Top 15 Features')
    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_fallback.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Feature importance plot saved")

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(f"\nDataset Information:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Total segments: {len(balanced_features)}")
print(f"  Analytes: {balanced_features['analyte_label'].unique().tolist()}")

print(f"\nPerformance Metrics:")
print(f"  Clustering ARI: {clustering_results['ari']:.3f}")
print(f"  Clustering NMI: {clustering_results['nmi']:.3f}")
print(f"  Binary accuracy (Event vs Baseline): {supervised_results['binary_accuracy']:.3f}")
print(f"  Multi-class accuracy: {supervised_results['multi_accuracy']:.3f}")
print(f"  Binary CV accuracy: {supervised_results['cv_scores'].mean():.3f} ± {supervised_results['cv_scores'].std():.3f}")

print(f"\nOutput Files:")
for file in output_dir.glob("*"):
    if file.is_file():
        print(f"  • {file.name}")

print(f"\nAnalysis complete! All results saved to: {output_dir.absolute()}")
print("="*60)